# Laboratory 5C — From the QHA model to observable properties

The final step is scientific validation. We compare the quasi-harmonic model
with independent thermodynamic, structural and elastic reference data.

Remember the rule for every plot:

> **First ask whether the quantities are thermodynamically comparable. Then ask
> about the absolute value. Finally ask about the trend.**


In [ ]:
import pandas as pd
from mineral_science_tools.qha_lab import (
    plot_thermochemistry,
    plot_alpha,
    plot_bulk_moduli,
    plot_room_temperature_compression,
    plot_pt_map,
)

thermo = pd.read_csv("data/mgo_nist_janaf_thermochemistry.csv")
volume_exp = pd.read_csv("data/mgo_reeber1995_volume_expansion.csv")
ks_exp = pd.read_csv("data/mgo_isaak1989_KS.csv")
eos_exp = pd.read_csv("data/mgo_speziale2001_300K_BM3.csv")


## 1. Entropy


In [ ]:
plot_thermochemistry("mgo_qha_poly_0GPa.hdf5", thermo, "S");


### Question 1

How well does QHA reproduce the magnitude and temperature trend of the standard
entropy? Is the remaining discrepancy large compared with the volume offset
seen in the previous notebook?

**Your answer:**  
_Write 3–4 lines here._


## 2. Constant-pressure heat capacity


In [ ]:
plot_thermochemistry("mgo_qha_poly_0GPa.hdf5", thermo, "Cp");


### Question 2 — What did QHA fix relative to Lab 4?

In Laboratory 4 the harmonic model produced $C_V$ at fixed volume. Here the QHA
produces $C_P$ because thermal expansion is no longer zero.

Does the comparison with calorimetry improve? Why can $C_P$ rise above the
harmonic Dulong–Petit value even though no new harmonic degrees of freedom have
appeared?

**Your answer:**  
_Write 4–5 lines here._


## 3. Enthalpy increment


In [ ]:
plot_thermochemistry("mgo_qha_poly_0GPa.hdf5", thermo, "H");


### Question 3

Why is $H(T)-H(298\,\mathrm K)$ often more useful for comparison than an
absolute first-principles enthalpy containing a very large electronic-energy
reference?

**Your answer:**  
_Write 2–3 lines here._


## 4. Thermal expansion

The NIST/REEBER table reports the **linear** expansion coefficient. MgO is
cubic, so the distributed CSV also contains the derived volumetric quantity
$\alpha_V=3\alpha_a$.


In [ ]:
plot_alpha(
    "mgo_qha_poly_0GPa.hdf5",
    volume_exp[volume_exp["T (K)"] <= 1500],
);


### Question 4

Does the QHA reproduce the sign and temperature evolution of $\alpha_V$?

Why would this entire property be identically zero in the strictly harmonic
model used in Laboratory 4?

**Your answer:**  
_Write 3–4 lines here._


## 5. Isothermal and adiabatic bulk modulus


In [ ]:
plot_bulk_moduli(
    "mgo_qha_poly_0GPa.hdf5",
    ks_exp[ks_exp["T (K)"] <= 1500],
);


### Question 5

The black experimental points are $K_S$, obtained from resonance/elastic-wave
measurements. Which theoretical curve should therefore be compared directly
with them: $K_T$ or $K_S$?

Does PBE predict a systematic stiffness offset? Even if it does, is the
**temperature derivative** physically reasonable?

**Your answer:**  
_Write 4–5 lines here._


## 6. Build a pressure–temperature result grid

So far we have looked at the ambient-pressure section. Now calculate one QHA
surface from 0 to 10 GPa and from 0 to 1500 K.

The region above roughly 1000 K is useful for observing trends, but structural
QHA predictions for MgO should be interpreted with increasing caution at high
temperature because explicit anharmonicity becomes progressively more
important.


In [ ]:
!quantas qha run mgo_qha.yaml \
    --scheme freq \
    --minimization poly \
    --thermal-expansion mixed_derivative \
    --temperature 0 1500 50 \
    --pressure 0 10 1 \
    --energy-degree 3 \
    --frequency-degree 3 \
    --output mgo_qha_poly_grid.hdf5 \
    --report mgo_qha_poly_grid.log \
    --no-progress --force


## 7. A 300 K compression section


In [ ]:
plot_room_temperature_compression(
    "mgo_qha_poly_grid.hdf5",
    eos_exp,
    temperature=300.0,
);


### Question 6

The black points are sampled from the experimental BM3 fit of Speziale et al.
(2001), not from individual raw diffraction measurements.

How does the theoretical compression **trend** compare with the experimental
EOS? Does the PBE volume offset remain visible under pressure?

**Your answer:**  
_Write 3–4 lines here._


## 8. Look at the full P–T response


In [ ]:
plot_pt_map("mgo_qha_poly_grid.hdf5", "volume");


In [ ]:
plot_pt_map("mgo_qha_poly_grid.hdf5", "alpha");


In [ ]:
plot_pt_map("mgo_qha_poly_grid.hdf5", "KT");


### Question 7 — Read the maps physically

From the three maps, state qualitatively what happens when:

1. temperature increases at constant pressure;
2. pressure increases at constant temperature.

Your answer should mention volume, thermal expansion and bulk modulus.

**Your answer:**  
_Write 5–6 lines here._


## 9. Final model assessment

The reference calculations use **PBE + POB-TZVP-REV2**. The basis is already a
large triple-zeta valence + polarization solid-state basis. Changing the
exchange–correlation functional — for example to a suitable hybrid — may change
structural and thermoelastic predictions, but at a substantially greater cost,
especially for a 128-atom phonon supercell repeated over seven volumes.

### Question 8 — Take-home message

Imagine that the calculated absolute volume and bulk modulus do not coincide
perfectly with experiment, but $V/V_0$, $\alpha_V(T)$, $C_P(T)$ and the pressure
and temperature trends are reproduced well.

Would you call the simulation unsuccessful? Explain what has been learned from
it and identify at least three distinct ways in which the remaining discrepancy
could be investigated.

**Your answer:**  
_Write 6–8 lines here._


## Before you finish

Your notebooks should now show the complete modelling chain:

```text
static DFT + phonons at several volumes
                ↓
             F(V,T)
                ↓
      equilibrium V(P,T)
                ↓
S, Cp, alphaV, KT, KS and structural response
                ↓
comparison with experiment
                ↓
assessment of trends, offsets and model limits
```

Save, commit, sync and verify your work on GitHub.
